# 18. 스태킹 하기 전에 다른 모델부터 테스트

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from lightgbm import LGBMRegressor
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42

train = pd.read_csv('../data/train.csv')
train = train.drop_duplicates(subset=[c for c in train.columns if c != 'ID']).reset_index(drop=True)
train['mean_working'] = train['mean_working'].fillna(0)
for col in ['medical_history', 'family_medical_history']:
    train[col] = train[col].fillna('None')
train['edu_level'] = train['edu_level'].fillna('Unknown')

# 기존 19개 파생변수 (00~06과 동일)
def add_features(df):
    data = df.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)
    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)
    data['anticipatory_stress'] = ((data['family_medical_history'] != 'None') & (data['medical_history'] == 'None')).astype(int)
    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)
    data['cardio_metabolic_load'] = data['map'] * data['bmi']
    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)
    return data

train = add_features(train)
activity_map = {'light': 0, 'moderate': 1, 'intense': 2}
edu_map = {'Unknown': 0, 'high school diploma': 1, 'bachelors degree': 2, 'graduate degree': 3}
train['activity'] = train['activity'].map(activity_map)
train['edu_level'] = train['edu_level'].map(edu_map)
tuned_params = dict(n_estimators=5000, learning_rate=0.03, num_leaves=127,
                     min_child_samples=10, random_state=RANDOM_STATE, verbose=-1)

## CatBoost 단독으로 해보기 (라벨인코딩 없이 카테고리 그대로)

In [1]:
cat_train = train.copy()
cat_cols = ['gender', 'activity', 'smoke_status', 'medical_history', 'family_medical_history',
            'sleep_pattern', 'edu_level']
for c in cat_cols:
    cat_train[c] = cat_train[c].astype(str)

x = cat_train.drop(columns=['ID', 'stress_score'])
y = cat_train['stress_score']
cat_idx = [x.columns.get_loc(c) for c in cat_cols]

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
maes = []
for tr_idx, va_idx in kf.split(x):
    model = CatBoostRegressor(iterations=3000, learning_rate=0.03, depth=6, loss_function='MAE',
                               random_seed=RANDOM_STATE, verbose=False, early_stopping_rounds=150)
    train_pool = Pool(x.iloc[tr_idx], y.iloc[tr_idx], cat_features=cat_idx)
    val_pool = Pool(x.iloc[va_idx], y.iloc[va_idx], cat_features=cat_idx)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)
    maes.append(mean_absolute_error(y.iloc[va_idx], model.predict(x.iloc[va_idx])))

print('CatBoost CV MAE:', np.mean(maes))

CatBoost CV MAE: 0.21296582757058786


LightGBM(0.174)보다 한참 나쁨. 폴드당 시간도 오래 걸림 (86초/fold). 여기서 접음.

## LGBM 5-seed 평균, XGBoost 단독, 둘 블렌딩

In [2]:
import json

def encode(df):
    d = df.copy()
    for col in ['gender', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern']:
        d[col] = LabelEncoder().fit_transform(d[col])
    return d

enc_train = encode(train)
x = enc_train.drop(columns=['ID', 'stress_score'])
y = enc_train['stress_score']

kf_fixed = KFold(n_splits=5, shuffle=True, random_state=42)
splits = list(kf_fixed.split(x))

seed_oofs = []
for seed in [42, 52, 62, 72, 82]:
    p = dict(tuned_params); p['random_state'] = seed
    oof = np.zeros(len(x))
    for tr_idx, va_idx in splits:
        m = LGBMRegressor(**p)
        m.fit(x.iloc[tr_idx], y.iloc[tr_idx], eval_set=[(x.iloc[va_idx], y.iloc[va_idx])],
              callbacks=[lgb.early_stopping(150, verbose=False)])
        oof[va_idx] = m.predict(x.iloc[va_idx])
    print(f'seed {seed}:', mean_absolute_error(y, oof))
    seed_oofs.append(oof)
lgbm_avg = np.mean(seed_oofs, axis=0)
print('5-seed 평균:', mean_absolute_error(y, lgbm_avg))

seed 42: 0.1719
seed 52: 0.1729
seed 62: 0.1728
seed 72: 0.1727
seed 82: 0.1728
5-seed 평균: 0.1720


In [3]:
xgb_params = dict(n_estimators=3000, learning_rate=0.03, max_depth=5, min_child_weight=5,
                   reg_alpha=0.1, reg_lambda=1.0, subsample=0.8, colsample_bytree=0.8,
                   random_state=42, tree_method='hist', early_stopping_rounds=150,
                   eval_metric='mae', verbosity=0)
oof_xgb = np.zeros(len(x))
for tr_idx, va_idx in splits:
    m = xgb.XGBRegressor(**xgb_params)
    m.fit(x.iloc[tr_idx], y.iloc[tr_idx], eval_set=[(x.iloc[va_idx], y.iloc[va_idx])], verbose=False)
    oof_xgb[va_idx] = m.predict(x.iloc[va_idx])
print('XGBoost:', mean_absolute_error(y, oof_xgb))

for w in [0.5, 0.7, 0.9, 1.0]:
    blend = w * lgbm_avg + (1 - w) * oof_xgb
    print(f'LGBM 비중 {w}:', mean_absolute_error(y, blend))

XGBoost: 0.1806
LGBM 비중 0.5: 0.1749
LGBM 비중 0.7: 0.1735
LGBM 비중 0.9: 0.1724
LGBM 비중 1.0: 0.1719


XGBoost도 LGBM보다 나쁘고, 섞을수록 오히려 점수가 나빠짐 (LGBM 100%가 제일 좋음).
다른 모델 섞는 건 의미 없어 보임. #07 스태킹이 안 됐던 것도 같은 이유인 듯.

## Optuna로 파라미터 다시 탐색 (기존엔 랜덤서치였음)

In [4]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def cv_mae(params, n_folds=5):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    maes = []
    for tr_idx, va_idx in kf.split(x):
        m = LGBMRegressor(**params)
        m.fit(x.iloc[tr_idx], y.iloc[tr_idx], eval_set=[(x.iloc[va_idx], y.iloc[va_idx])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
        maes.append(mean_absolute_error(y.iloc[va_idx], m.predict(x.iloc[va_idx])))
    return np.mean(maes)

def objective(trial):
    params = dict(
        n_estimators=3000,
        learning_rate=trial.suggest_float('learning_rate', 0.012, 0.04, log=True),
        num_leaves=trial.suggest_int('num_leaves', 60, 150),
        max_depth=-1,
        min_child_samples=trial.suggest_int('min_child_samples', 3, 25),
        reg_alpha=trial.suggest_float('reg_alpha', 1e-4, 0.05, log=True),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 0.2, log=True),
        subsample=trial.suggest_float('subsample', 0.8, 1.0),
        subsample_freq=1,
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.75, 1.0),
        random_state=42,
        verbose=-1,
    )
    return cv_mae(params)

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.enqueue_trial({'learning_rate': 0.03, 'num_leaves': 127, 'min_child_samples': 10,
                     'reg_alpha': 0.1, 'reg_lambda': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8})
study.optimize(objective, n_trials=80)
print('best MAE:', study.best_value)
print('best params:', study.best_params)

with open('optuna_round2_best_params.json', 'w') as f:
    p = dict(study.best_params)
    p.update(n_estimators=3000, max_depth=-1, subsample_freq=1, random_state=42, verbose=-1)
    json.dump(p, f)

best MAE: 0.1719950718987654
best params: {'learning_rate': 0.0226, 'num_leaves': 108, 'min_child_samples': 7, 'reg_alpha': 0.00255,
'reg_lambda': 0.0319, 'subsample': 0.963, 'colsample_bytree': 0.959}


좁혀서 다시 돌려도 같은 지점으로 수렴함. 진짜 최적점 근처인 듯.

## stress_score가 0.01 단위 그리드라서 반올림해보기

In [5]:
print(train['stress_score'].nunique())

best_params = json.load(open('optuna_round2_best_params.json'))
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(x))
for tr_idx, va_idx in kf.split(x):
    m = LGBMRegressor(**best_params)
    m.fit(x.iloc[tr_idx], y.iloc[tr_idx], eval_set=[(x.iloc[va_idx], y.iloc[va_idx])],
          callbacks=[lgb.early_stopping(150, verbose=False)])
    oof[va_idx] = m.predict(x.iloc[va_idx])

print('원본:', mean_absolute_error(y, oof))
print('반올림:', mean_absolute_error(y, np.clip(np.round(oof, 2), 0, 1)))

101
원본: 0.1719
반올림: 0.1718


거의 차이 없음. 그래도 손해는 없으니 최종 제출에는 반올림 적용.

In [6]:
import os

test = pd.read_csv('../data/test.csv')
test['mean_working'] = test['mean_working'].fillna(0)
for col in ['medical_history', 'family_medical_history']:
    test[col] = test[col].fillna('None')
test['edu_level'] = test['edu_level'].fillna('Unknown')
test = add_features(test)
test['activity'] = test['activity'].map(activity_map)
test['edu_level'] = test['edu_level'].map(edu_map)

full_x = x.copy()
full_test = test.drop(columns=['ID']).copy()
for col in ['gender', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern']:
    le = LabelEncoder().fit(train[col])
    unseen = [l for l in np.unique(full_test[col]) if l not in le.classes_]
    if unseen:
        le.classes_ = np.append(le.classes_, unseen)
    full_test[col] = le.transform(full_test[col])

kf = KFold(n_splits=5, shuffle=True, random_state=42)
test_pred = np.zeros(len(full_test))
for tr_idx, va_idx in kf.split(full_x):
    m = LGBMRegressor(**best_params)
    m.fit(full_x.iloc[tr_idx], y.iloc[tr_idx], eval_set=[(full_x.iloc[va_idx], y.iloc[va_idx])],
          callbacks=[lgb.early_stopping(150, verbose=False)])
    test_pred += np.clip(m.predict(full_test), 0, 1) / kf.n_splits

sample_submission = pd.read_csv('../data/sample_submission.csv')
os.makedirs('../submissions', exist_ok=True)
sample_submission['stress_score'] = np.clip(np.round(test_pred, 2), 0, 1)
sample_submission.to_csv('../submissions/submit_18_final_rounded.csv', index=False)
print('저장 완료')

저장 완료
